<a href="https://colab.research.google.com/github/a80651083-crypto/MyFirstAI/blob/main/Llama%203.2%203B%20Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Cell 1 — Install Dependencies
!pip install -q transformers trl peft accelerate bitsandbytes datasets wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00


In [3]:
#Cell 2 — Login
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [15]:
#Cell 3 — Load Model in 4-bit (fits T4)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for training + attach LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready! ✅")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713
Model ready! ✅


In [17]:
#Cell 4 — Define Reward Function
import re

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        output = completion[0]["content"] if isinstance(completion[0], dict) else completion[0]
        score = 0.0

        # ── Length reward ──────────────────────────────
        # Too short = useless, too long = rambling
        length = len(output.split())
        if 50 < length < 300: score += 0.5
        elif length >= 300: score += 0.2   # penalize rambling

        # ── Code quality reward ────────────────────────
        # Rewards responses that include actual code blocks
        code_blocks = re.findall(r"```[\s\S]*?```", output)
        if code_blocks:
            score += 0.5
            # Bonus if code block has actual content (not empty)
            if any(len(cb) > 20 for cb in code_blocks): score += 0.3

        # ── Structure reward ───────────────────────────
        # Rewards clear step-by-step explanations
        has_steps = bool(re.search(r"\b(step \d|first|then|finally|\d\.)\b", output.lower()))
        if has_steps: score += 0.3

        # ── Relevance reward ───────────────────────────
        # Checks if response addresses the prompt topic
        prompt_words = set(prompt.lower().split())
        output_words = set(output.lower().split())
        overlap = len(prompt_words & output_words) / max(len(prompt_words), 1)
        score += overlap * 0.5

        # ── Penalize bad responses ─────────────────────
        # Penalize if model refuses or says it can't help
        refusal_phrases = ["i cannot", "i can't", "i am not able", "as an ai"]
        if any(p in output.lower() for p in refusal_phrases): score -= 1.0

        # Penalize repetition
        words = output.lower().split()
        unique_ratio = len(set(words)) / max(len(words), 1)
        if unique_ratio < 0.4: score -= 0.5   # too repetitive

        rewards.append(max(score, 0.0))  # no negative rewards
    return rewards

In [18]:
# Cell 5 — GRPO Training (Proper Dataset)
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

data = [
    # Python basics
    {"prompt": "How do I create a Python function?"},
    {"prompt": "How do I use a for loop in Python?"},
    {"prompt": "How do I read a file in Python?"},
    {"prompt": "How do I handle errors in Python using try/except?"},
    {"prompt": "How do I use list comprehension in Python?"},
    {"prompt": "What is the difference between a list and a dictionary in Python?"},
    {"prompt": "How do I install a Python package using pip?"},
    {"prompt": "How do I use classes in Python?"},
    {"prompt": "How do I import a module in Python?"},
    {"prompt": "How do I write a Python script that takes command line arguments?"},

    # AI / ML
    {"prompt": "What is reinforcement learning?"},
    {"prompt": "What is the difference between supervised and unsupervised learning?"},
    {"prompt": "What is a neural network?"},
    {"prompt": "What is a reward function in reinforcement learning?"},
    {"prompt": "What is GRPO and how does it work?"},
    {"prompt": "What is LoRA and why is it used for fine-tuning?"},
    {"prompt": "What is the difference between a language model and a world model?"},
    {"prompt": "What is overfitting and how do you prevent it?"},
    {"prompt": "What is gradient descent?"},
    {"prompt": "What is the transformer architecture?"},

    # Linux / Terminal
    {"prompt": "How do I navigate folders in Linux terminal?"},
    {"prompt": "How do I install a package in Linux using apt?"},
    {"prompt": "How do I check disk space in Linux?"},
    {"prompt": "How do I kill a running process in Linux?"},
    {"prompt": "How do I set file permissions in Linux?"},
    {"prompt": "How do I check what's running on a port in Linux?"},
    {"prompt": "How do I write a bash script?"},
    {"prompt": "How do I use grep to search text in Linux?"},
    {"prompt": "How do I connect to a remote server using SSH?"},
    {"prompt": "How do I check RAM and CPU usage in Linux?"},

    # Android / App Dev
    {"prompt": "How does Android Accessibility Service work?"},
    {"prompt": "How do I create an overlay window in Android?"},
    {"prompt": "What is an APK file?"},
    {"prompt": "How do I debug an Android app?"},
    {"prompt": "What is the difference between an Activity and a Service in Android?"},
    {"prompt": "How do I make an API call in an Android app?"},
    {"prompt": "How do I store data locally in an Android app?"},
    {"prompt": "What is Android Jetpack?"},
    {"prompt": "How do I request permissions in Android?"},
    {"prompt": "What is the difference between Java and Kotlin for Android?"},

    # APIs / Backend
    {"prompt": "What is a REST API?"},
    {"prompt": "How do I make an API call using Python?"},
    {"prompt": "What is JSON and how do I parse it in Python?"},
    {"prompt": "What is the difference between GET and POST requests?"},
    {"prompt": "How do I create a simple Python web server using Flask?"},

    # Git / GitHub
    {"prompt": "How do I push code to GitHub?"},
    {"prompt": "How do I create a new branch in Git?"},
    {"prompt": "How do I fix a merge conflict in Git?"},
    {"prompt": "What is the difference between git pull and git fetch?"},
    {"prompt": "How do I undo my last commit in Git?"},

    # Java
    {"prompt": "How do I create a class in Java?"},
    {"prompt": "What is the difference between Java and Python?"},
    {"prompt": "How do I handle exceptions in Java?"},
    {"prompt": "What is object-oriented programming in Java?"},
    {"prompt": "How do I create an ArrayList in Java?"},
    {"prompt": "What is the difference between an interface and an abstract class in Java?"},
    {"prompt": "How do I read user input in Java?"},
    {"prompt": "What is a constructor in Java?"},
    {"prompt": "How do I loop through an array in Java?"},
    {"prompt": "What is the difference between == and .equals() in Java?"},
]

dataset = Dataset.from_list(data)

config = GRPOConfig(
    output_dir="./grpo-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    logging_steps=5,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=reward_fn,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
print("Training done! ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
5,0.000000
10,0.000000
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000
40,0.000000
45,0.000000
50,0.000000


Step,Training Loss
5,0.000000
10,0.000000
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000
40,0.000000
45,0.000000
50,0.000000


Training done! ✅


In [19]:
# Cell 6 upload HuggingFace
model.push_to_hub("Aion2/llama3.2-3b-grpo-v1")
tokenizer.push_to_hub("Aion2/llama3.2-3b-grpo-v1")
print("Uploaded! ✅")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  23%|##2       | 1.04MB / 4.60MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpnjk8pxis/tokenizer.json:  46%|####6     | 7.96MB / 17.2MB            

Uploaded! ✅


In [20]:
# Cell 7 — Chat Test (Inference)
from transformers import GenerationConfig

def chat(message):
    # Apply proper chat template
    messages = [{"role": "user", "content": message}]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generate with repetition penalty to stop loops
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.3,  # ← fixes the "QuestionQuestion" loop
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode only the new tokens (not the prompt)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"👤 User: {message}")
    print(f"🤖 Assistant: {response}")
    print("---")

chat("How do I create a Python function?")
chat("What is reinforcement learning?")
chat("How do I install a package in Linux?")

👤 User: How do I create a Python function?
🤖 Assistant: CreatingQuestion question##QuestionQuestion QuestionQuestion question ProblemQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestion QuestionsQuestionQuestion question daysTags questionQuestionQuestionQuestionQuestionimport的在QuestionQuestion有QuestionQuestionQuestionQuestionQuestion questionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestion QuestionQuestion QuestionThe studyQuestionQuestionQuestion用QuestionQuestionQuestionQuestionQuestionQuestionQuestion questionQuestiondef#QuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestion questionQuestionQuestion questionHomeQuestion Question下QuestionQuestionQuestionQuestionQuestion Question#QuestionQuestionQuestionQuestionQuestionQuestion questions is was wouldnThe将QuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuestionQuest